In [1]:
# ============================================================
# Cell 1: Setup
# ============================================================
import json
from pathlib import Path

# Mount Drive in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Standard tuning, MIDI of each open string
# Index 0 = low E (string 6 in guitarist notation)
# Index 5 = high E (string 1)
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
MAX_FRET = 24

# Update if your JAMS folder lives elsewhere
JAMS_DIR = Path('/content/drive/MyDrive/Capstone/GuitarSet/Annotations')

Mounted at /content/drive


In [2]:
# ============================================================
# Cell 2: JAMS parser
# ============================================================
# Lightweight raw-JSON parser. Avoids the jams library dependency
# so this notebook is self-contained. Returns the same shape as the
# eval pipeline loader plus the raw beat grid (which we need for
# rendering).

def _annotation_rows(annotation):
    """Normalize JAMS annotation data into a list of row dicts.
    JAMS files use either a list-of-dicts format or a dict-of-lists
    format depending on tooling version — handle both."""
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []


def _string_index_from_source(data_source):
    """GuitarSet stores one note_midi annotation per string with
    data_source = '0'..'5' (0 = low E)."""
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except (TypeError, ValueError):
        return None


def parse_jams(jams_path):
    """Parse a GuitarSet JAMS file into a structured dict.

    Returns:
        dict with keys:
            recording: filename stem
            notes:  list of {start, duration, midi, string, fret}
            beats:  list of beat times (float seconds)
            chords: list of {start, duration, end, chord}
            tempo:  float BPM or None
            key:    string like 'Eb:major' or None
    """
    jams_path = Path(jams_path)
    with open(jams_path) as f:
        jam = json.load(f)

    notes, chords, beats = [], [], []
    tempo = None
    key = None

    for ann in jam.get('annotations', []):
        ns = ann.get('namespace', '')
        rows = _annotation_rows(ann)
        ds = ann.get('annotation_metadata', {}).get('data_source', '')

        if ns == 'note_midi':
            string_idx = _string_index_from_source(ds)
            if string_idx is None:
                continue
            for r in rows:
                v = r.get('value')
                # Some JAMS variants nest the pitch under a dict
                if isinstance(v, dict):
                    midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                else:
                    midi = v
                if midi is None:
                    continue
                midi_int = int(round(float(midi)))
                fret = midi_int - OPEN_STRING_MIDI[string_idx]
                if fret < 0 or fret > MAX_FRET:
                    continue
                notes.append({
                    'start':    float(r.get('time', 0.0)),
                    'duration': float(r.get('duration', 0.0) or 0.0),
                    'midi':     midi_int,
                    'string':   string_idx,
                    'fret':     int(fret),
                })

        elif ns in ('chord', 'chord_harte'):
            for r in rows:
                start = float(r.get('time', 0.0))
                duration = float(r.get('duration', 0.0) or 0.0)
                chords.append({
                    'start': start, 'duration': duration,
                    'end': start + duration, 'chord': r.get('value'),
                })

        elif ns in ('beat', 'beat_position'):
            for r in rows:
                beats.append(float(r.get('time', 0.0)))

        elif ns == 'key_mode':
            if rows:
                key = rows[0].get('value')

        elif ns == 'tempo':
            if rows:
                tempo = float(rows[0].get('value', 0.0)) or None

    # Tempo from filename as a fallback. GuitarSet filenames look
    # like "00_BN1-129-Eb_comp" — the second token has the BPM.
    if tempo is None:
        try:
            tempo = float(jams_path.stem.split('_')[1].split('-')[1])
        except (IndexError, ValueError):
            pass

    notes.sort(key=lambda n: (n['start'], n['midi']))
    beats.sort()
    chords.sort(key=lambda c: c['start'])

    return {
        'recording': jams_path.stem,
        'notes': notes,
        'beats': beats,
        'chords': chords,
        'tempo': tempo,
        'key': key,
    }

In [3]:
# ============================================================
# Cell 3: ASCII tab renderer
# ============================================================

# Display order: high E on top, low E on bottom (standard tab convention)
# DISPLAY_TO_STRING[i] = GuitarSet string index at display row i
DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]
STRING_LABELS    = ['e', 'B', 'G', 'D', 'A', 'E']


def _build_time_grid(beats, tempo, end_time, subdivisions_per_beat):
    """Build the list of time points that define tab columns.

    Prefer JAMS beat annotations (they reflect actual musical beats,
    not assumed metronome time). Subdivide each beat into N steps for
    finer resolution. Fall back to tempo if no beats. Final fallback
    is a fixed 0.25s grid, which is rough but produces output for
    files with no metric annotations.
    """
    if beats and len(beats) >= 2:
        grid = []
        for i in range(len(beats) - 1):
            step = (beats[i + 1] - beats[i]) / subdivisions_per_beat
            for j in range(subdivisions_per_beat):
                grid.append(beats[i] + j * step)
        # Extend past the last beat using the previous gap
        last_step = (beats[-1] - beats[-2]) / subdivisions_per_beat
        while grid[-1] < end_time:
            grid.append(grid[-1] + last_step)
        return grid

    if tempo and tempo > 0:
        step = 60.0 / tempo / subdivisions_per_beat
        n_steps = int(end_time / step) + subdivisions_per_beat
        return [i * step for i in range(n_steps)]

    return [i * 0.25 for i in range(int(end_time / 0.25) + 2)]


def render_ascii_tab(parsed, subdivisions_per_beat=2, beats_per_measure=4,
                     measures_per_line=4, col_width=3, max_notes=None):
    """Render a parsed GuitarSet recording as ASCII tab.

    Args:
        parsed: dict from parse_jams (must contain 'notes' and 'beats').
        subdivisions_per_beat: tab columns per beat. 2 = eighth notes,
            4 = sixteenth notes. Higher resolution = wider output.
        beats_per_measure: typically 4 for 4/4.
        measures_per_line: how many measures to show per line of tab
            before wrapping.
        col_width: characters per column. 3 fits two-digit frets cleanly.
        max_notes: if set, truncate to this many notes (useful for
            previewing long recordings).

    Returns:
        ASCII tab as a single multi-line string.
    """
    notes = parsed['notes']
    if max_notes is not None:
        notes = notes[:max_notes]
    if not notes:
        return '(no notes in this recording)'

    end_time = max(n['start'] + n['duration'] for n in notes) + 0.5
    grid = _build_time_grid(
        parsed.get('beats', []), parsed.get('tempo'),
        end_time, subdivisions_per_beat,
    )
    n_cols = len(grid)

    # cells[display_row][col] = fret number, or None for empty
    cells = [[None] * n_cols for _ in range(6)]
    collisions = 0

    for note in notes:
        # Snap to nearest grid column
        col = min(range(n_cols), key=lambda i: abs(grid[i] - note['start']))
        try:
            row = DISPLAY_TO_STRING.index(note['string'])
        except ValueError:
            continue
        if cells[row][col] is not None:
            collisions += 1
        cells[row][col] = note['fret']

    # Format each cell as `col_width` chars, fret left-aligned, dashes after
    def fmt(v):
        if v is None:
            return '-' * col_width
        s = str(v)
        if len(s) >= col_width:
            return s[:col_width]
        return s + '-' * (col_width - len(s))

    formatted = [[fmt(cells[r][c]) for c in range(n_cols)] for r in range(6)]

    # Assemble output with measure bars and line wrapping
    cols_per_measure = beats_per_measure * subdivisions_per_beat
    cols_per_line = cols_per_measure * measures_per_line
    lines = []

    for start in range(0, n_cols, cols_per_line):
        end = min(start + cols_per_line, n_cols)
        for row in range(6):
            parts = []
            for c in range(start, end):
                if c > start and (c - start) % cols_per_measure == 0:
                    parts.append('|')
                parts.append(formatted[row][c])
            lines.append(f"{STRING_LABELS[row]}|{''.join(parts)}|")
        lines.append('')

    out = '\n'.join(lines)
    if collisions:
        out += f"\n[note: {collisions} cell collisions — multiple notes quantized to the same string/column]"
    return out

In [4]:
# ============================================================
# Cell 4: Demo
# ============================================================

# Pick any GuitarSet JAMS file
jams_files = sorted(JAMS_DIR.glob('*.jams'))
print(f"Found {len(jams_files)} JAMS files in {JAMS_DIR}")

target = jams_files[0]   # e.g. 00_BN1-129-Eb_comp.jams
parsed = parse_jams(target)

print(f"\nRecording: {parsed['recording']}")
print(f"  Tempo:  {parsed['tempo']}")
print(f"  Key:    {parsed['key']}")
print(f"  Notes:  {len(parsed['notes'])}")
print(f"  Beats:  {len(parsed['beats'])}")
print(f"  Chords: {len(parsed['chords'])}")
print()

# Render the first ~16 measures so the output is browsable in Colab
tab = render_ascii_tab(
    parsed,
    subdivisions_per_beat=2,
    beats_per_measure=4,
    measures_per_line=4,
    max_notes=120,
)
print(tab)

Found 9 JAMS files in /content/drive/MyDrive/Capstone/GuitarSet/Annotations

Recording: 00_BN1-129-Eb_comp
  Tempo:  129.0
  Key:    Eb:major
  Notes:  133
  Beats:  48
  Chords: 12

e|------------------------|------------------------|8-----------6--------6--|------------------------|
B|6--------------8--------|6--------8--------------|6-----------6--------8--|---------6--------------|
G|7--------------7--------|7--7-----7-----7-----7--|---7-----7-----7-----7--|---------7--------------|
D|---------8-----------8--|---------------8--------|8-----------8-----------|---8-----------8-----6--|
A|6-----6-----6-----------|6-----6--6--------6-----|------6--------------6--|------6-----------------|
E|------------------------|------------------------|------------------------|------------------------|

e|------------------------|---8--------8-----------|8--------------------6--|------------------------|
B|6--------6-----------6--|---8--------------------|8--------8-----------8--|---8--------------